In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent / "src"))


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ast import literal_eval

from data import GDPHMM_OAR_V3, GDPHMM_DATA_BASE_PATH


In [23]:
metadata = pd.read_csv('../data/gdphmm-hn/incoming/train_metadata.csv')
for col in metadata.columns[metadata.columns.str.endswith(('_shape', '_range', '_bbox'))]:
  metadata[col] = metadata[col].apply(lambda x: literal_eval(x) if isinstance(x, str) else None)

metadata['all_mask'] = metadata['all_mask'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)

filtered_metadata = metadata[~metadata['sample_name'].isin({'HN-CHUS-100+S215Ag+MOS_4436', 'HN-CHUS-100+S29Ag+MOS_4436'})]
filtered_metadata = filtered_metadata[filtered_metadata['isVMAT']]
len(filtered_metadata)

472

In [42]:
filtered_metadata['subject_id'] = filtered_metadata['sample_name'].str.split('+').str[0]
subject_ids = filtered_metadata['subject_id']

In [25]:
patient_metadata = []

In [26]:
patient_metadata_hnscc_file = GDPHMM_DATA_BASE_PATH / 'gdphmm-hn' / 'incoming' / 'patient-metadata' / 'HNSCC.csv'
patient_hnscc_metadata = pd.read_csv(patient_metadata_hnscc_file)
patient_hnscc_metadata['sample_name'] = patient_hnscc_metadata['TCIA Radiomics dummy ID of To_Submit_Final']
patient_hnscc_metadata


,TCIA Radiomics dummy ID of To_Submit_Final,Gender,Age at Diag,Smoking status,Smoking status (Packs-Years),Tumor laterality,Cancer subsite of origin,HPV Status,T-category,N-category,...,Regional control_duration of Merged updated ASRM V2,Locoregional control,Locoregional control_duration of Merged updated ASRM V2,Freedom from distant metastasis,Freedom from distant metastasis_duration of Merged updated ASRM V2,Relapse-free survival,Days to last FU,Neck Dissection after IMRT,TCIA Radiomics dummy ID of Merged updated ASRM V2,sample_name
0,HNSCC-01-0003,Male,48,Former,5.0,R,Tonsil,Unknown,2,2,...,3142.0,No,1106.0,Yes,3142.0,No,3142,No,HNSCC-01-0003,HNSCC-01-0003
1,HNSCC-01-0024,Female,80,Never,0.0,L,Soft palate,Unknown,3,2,...,3405.0,No,1886.0,Yes,3405.0,No,3405,No,HNSCC-01-0024,HNSCC-01-0024
2,HNSCC-01-0026,Male,59,Never,0.0,R,Base of tongue,Unknown,2,2,...,1907.0,Yes,1907.0,Yes,1907.0,Yes,1907,Yes,HNSCC-01-0026,HNSCC-01-0026
3,HNSCC-01-0030,Male,63,Never,0.0,R,Base of tongue,Unknown,4,1,...,2966.0,Yes,2966.0,Yes,2966.0,Yes,2966,No,HNSCC-01-0030,HNSCC-01-0030
4,HNSCC-01-0033,Female,44,Current,30.0,Midline,NOS,Unknown,3,0,...,3693.0,Yes,3693.0,Yes,3693.0,Yes,3693,No,HNSCC-01-0033,HNSCC-01-0033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487,HNSCC-01-0626,Male,60,Current,0.0,L,Base of tongue,P,2,2,...,1738.0,Yes,1738.0,Yes,1738.0,Yes,1738,No,HNSCC-01-0626,HNSCC-01-0626
488,HNSCC-01-0627,Male,65,Former,8.0,R,Base of tongue,P,1,2,...,1835.0,Yes,1835.0,Yes,1835.0,Yes,1835,No,HNSCC-01-0627,HNSCC-01-0627
489,HNSCC-01-0628,Female,69,Current,35.0,Midline,Base of tongue,P,4,2,...,224.0,Yes,224.0,Yes,224.0,Yes,224,No,HNSCC-01-0628,HNSCC-01-0628
490,HNSCC-01-0629,Female,63,Former,6.0,R,Base of tongue,N,4,2,...,766.0,Yes,766.0,Yes,766.0,Yes,766,No,HNSCC-01-0629,HNSCC-01-0629


In [27]:
d = patient_hnscc_metadata[patient_hnscc_metadata['sample_name'].isin(subject_ids)]
d = d.rename(columns={
  'Age at Diag': 'age',
  'Gender': 'sex',
  'Smoking status': 'smoking_status',
})
d['sex'] = d['sex'].map({'Male': 'male', 'Female': 'female'})
patient_metadata.append(d[['sample_name', 'age', 'sex', 'smoking_status']])

In [28]:
patient_metadata_2_file = GDPHMM_DATA_BASE_PATH / 'gdphmm-hn' / 'incoming' / 'patient-metadata' / 'Head-Neck-PET-CT.xlsx'
patient_metadata_hgj = pd.read_excel(patient_metadata_2_file, sheet_name='HGJ')
patient_metadata_chum = pd.read_excel(patient_metadata_2_file, sheet_name='CHUM')
patient_metadata_hmr = pd.read_excel(patient_metadata_2_file, sheet_name='HMR')
patient_metadata_chus = pd.read_excel(patient_metadata_2_file, sheet_name='CHUS')


In [29]:
d = patient_metadata_hgj[patient_metadata_hgj['Patient #'].isin(subject_ids)]
d = d.rename(columns={
  'Patient #': 'sample_name',
  'Age': 'age',
  'Sex': 'sex',
})
d['sex'] = d['sex'].map({'M': 'male', 'F': 'female'})
patient_metadata.append(d[['sample_name', 'age', 'sex']])

In [30]:
d = patient_metadata_chum[patient_metadata_chum['Patient #'].isin(subject_ids)]
d = d.rename(columns={
  'Patient #': 'sample_name',
  'Age': 'age',
  'Sex': 'sex',
})
d['sex'] = d['sex'].map({'M': 'male', 'F': 'female'})
patient_metadata.append(d[['sample_name', 'age', 'sex']])

In [31]:
d = patient_metadata_hmr[patient_metadata_hmr['Patient #'].isin(subject_ids)]
d = d.rename(columns={
  'Patient #': 'sample_name',
  'Age': 'age',
  'Sex': 'sex',
})
d['sex'] = d['sex'].map({'M': 'male', 'F': 'female'})
patient_metadata.append(d[['sample_name', 'age', 'sex']])

In [32]:
d = patient_metadata_chus[patient_metadata_chus['Patient #'].isin(subject_ids)]
d = d.rename(columns={
  'Patient #': 'sample_name',
  'Age': 'age',
  'Sex': 'sex',
})
d['sex'] = d['sex'].map({'M': 'male', 'F': 'female'})
patient_metadata.append(d[['sample_name', 'age', 'sex']])

In [37]:
patient_metadata = pd.concat(patient_metadata, ignore_index=True).set_index('sample_name')
patient_metadata

,age,sex,smoking_status
sample_name,,,
HNSCC-01-0003,48.0,male,Former
HNSCC-01-0026,59.0,male,Never
HNSCC-01-0030,63.0,male,Never
HNSCC-01-0033,44.0,female,Current
HNSCC-01-0034,48.0,male,Current
...,...,...,...
HN-CHUS-064,66.0,male,NaN
HN-CHUS-089,65.0,male,NaN
HN-CHUS-096,67.0,male,NaN


In [43]:
filtered_metadata[~filtered_metadata['subject_id'].isin(patient_metadata.index)]

,sample_name,Trachea_shape,Trachea_range,Trachea_bbox,Thyroid-PTV_shape,Thyroid-PTV_range,Thyroid-PTV_bbox,Thyroid_shape,Thyroid_range,Thyroid_bbox,...,PTVLowOPT_shape,PTVLowOPT_range,PTVLowOPT_bbox,PTVLow-PTVMid_shape,PTVLow-PTVMid_range,PTVLow-PTVMid_bbox,PTVLow_shape,PTVLow_range,PTVLow_bbox,subject_id
2,HNSCC-01-0054+A4Ac+MOS_0511,None,None,None,"(128, 101, 194)","(0, 1)","(12, 31, 84, 24, 11, 21)","(128, 101, 194)","(0, 1)","(12, 31, 84, 24, 11, 21)",...,"(128, 101, 194)","(0, 1)","(45, 33, 67, 51, 26, 46)","(128, 101, 194)","(0, 1)","(45, 33, 67, 51, 26, 46)","(128, 101, 194)","(0, 1)","(45, 33, 67, 51, 26, 46)",HNSCC-01-0054
3,HNC_171+A4Ac+MOS_31950,"(128, 109, 193)","(0, 1)","(0, 39, 89, 47, 24, 13)","(128, 109, 193)","(0, 1)","(32, 41, 91, 15, 6, 14)","(128, 109, 193)","(0, 1)","(31, 41, 89, 16, 7, 17)",...,"(128, 109, 193)","(0, 1)","(23, 22, 63, 79, 47, 67)","(128, 109, 193)","(0, 1)","(23, 22, 63, 79, 47, 67)","(128, 109, 193)","(0, 1)","(23, 22, 63, 79, 47, 67)",HNC_171
4,HNC_027+A4Ac+MOS_17377,"(132, 111, 180)","(0, 1)","(0, 34, 85, 30, 21, 11)","(132, 111, 180)","(0, 1)","(14, 34, 81, 22, 13, 20)","(132, 111, 180)","(0, 1)","(14, 34, 80, 24, 13, 22)",...,"(132, 111, 180)","(0, 1)","(17, 8, 50, 108, 62, 68)","(132, 111, 180)","(0, 1)","(17, 8, 50, 108, 62, 68)","(132, 111, 180)","(0, 1)","(17, 8, 50, 108, 62, 68)",HNC_027
5,HNC_107+A4Ac+MOS_49283,"(128, 113, 193)","(0, 1)","(0, 33, 89, 46, 27, 12)","(128, 113, 193)","(0, 1)","(30, 32, 87, 25, 12, 17)","(128, 113, 193)","(0, 1)","(30, 32, 86, 27, 12, 19)",...,"(128, 113, 193)","(0, 1)","(21, 16, 71, 94, 55, 67)","(128, 113, 193)","(0, 1)","(21, 16, 71, 94, 55, 67)","(128, 113, 193)","(0, 1)","(21, 16, 71, 94, 55, 67)",HNC_107
16,HNSCC-01-0109+A4Ac+MOS_42365,"(127, 104, 200)","(0, 1)","(0, 33, 96, 23, 21, 10)","(127, 104, 200)","(0, 1)","(13, 35, 90, 17, 11, 23)","(127, 104, 200)","(0, 1)","(13, 35, 90, 17, 11, 23)",...,"(127, 104, 200)","(0, 1)","(41, 22, 83, 54, 45, 49)","(127, 104, 200)","(0, 1)","(41, 22, 83, 54, 44, 49)","(127, 104, 200)","(0, 1)","(41, 22, 83, 54, 44, 49)",HNSCC-01-0109
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1393,HNSCC-01-0214+A4Ac+MOS_28787,"(128, 111, 200)","(0, 1)","(2, 50, 85, 39, 22, 23)",None,None,None,"(128, 111, 200)","(0, 1)","(56, 43, 92, 16, 10, 7)",...,"(128, 111, 200)","(0, 1)","(95, 37, 79, 16, 28, 16)","(128, 111, 200)","(0, 1)","(95, 37, 79, 16, 28, 16)","(128, 111, 200)","(0, 1)","(95, 37, 79, 16, 28, 16)",HNSCC-01-0214
1394,HNC_135+A4Ac+MOS_2696,"(128, 122, 218)","(0, 1)","(0, 33, 101, 45, 29, 12)","(128, 122, 218)","(0, 1)","(30, 34, 99, 19, 12, 18)","(128, 122, 218)","(0, 1)","(30, 34, 96, 25, 12, 23)",...,"(128, 122, 218)","(0, 1)","(22, 14, 65, 74, 59, 67)","(128, 122, 218)","(0, 1)","(22, 14, 65, 74, 59, 67)","(128, 122, 218)","(0, 1)","(22, 14, 65, 74, 59, 67)",HNC_135
1397,HNSCC-01-0117+A4Ac+MOS_20885,"(128, 96, 165)","(0, 1)","(0, 44, 77, 32, 14, 11)","(128, 96, 165)","(0, 1)","(16, 43, 73, 26, 11, 19)","(128, 96, 165)","(0, 1)","(16, 43, 73, 29, 11, 19)",...,"(128, 96, 165)","(0, 1)","(44, 20, 63, 46, 46, 37)","(128, 96, 165)","(0, 1)","(44, 20, 63, 46, 46, 38)","(128, 96, 165)","(0, 1)","(44, 20, 63, 46, 46, 38)",HNSCC-01-0117
1403,HNC_009+A4Ac+MOS_51704,"(128, 111, 198)","(0, 1)","(0, 38, 91, 46, 24, 11)","(128, 111, 198)","(0, 1)","(33, 38, 88, 14, 12, 19)","(128, 111, 198)","(0, 1)","(33, 38, 87, 15, 12, 21)",...,"(128, 111, 198)","(0, 1)","(35, 42, 54, 71, 32, 58)","(128, 111, 198)","(0, 1)","(35, 42, 54, 71, 32, 58)","(128, 111, 198)","(0, 1)","(35, 42, 54, 71, 32, 58)",HNC_009
